In [0]:
from pyspark.sql import functions as F
scores = spark.table("workspace.mortgage_risk.test_scores_gbt_v1")

In [0]:
# Prepare both populations
band_scores = scores.withColumn(
    "population", F.lit("All eligible loans")
)

healthy_scores = scores.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population", F.lit("Current status 00")
)

band_scores = band_scores.unionByName(healthy_scores)

In [0]:
# Group observations by calibrated risk
band_scores = band_scores.withColumn(
    "risk_band",
    F.when(F.col("calibrated_risk") < 0.002, "01: Below 0.2%")
     .when(F.col("calibrated_risk") < 0.005, "02: 0.2% to below 0.5%")
     .when(F.col("calibrated_risk") < 0.01, "03: 0.5% to below 1%")
     .when(F.col("calibrated_risk") < 0.02, "04: 1% to below 2%")
     .when(F.col("calibrated_risk") < 0.05, "05: 2% to below 5%")
     .when(F.col("calibrated_risk") < 0.10, "06: 5% to below 10%")
     .otherwise("07: 10% or more")
)

In [0]:
# Compare predicted and observed rates
band_check = band_scores.groupBy("population", "risk_band").agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.avg("calibrated_risk").alias("predicted_rate"),
    F.avg("label_12m").alias("observed_rate")
)

band_check = band_check.withColumn(
    "gap_percentage_points",
    (F.col("predicted_rate") - F.col("observed_rate")) * 100
)

In [0]:
display(
    band_check.select(
        "population",
        "risk_band",
        "rows",
        "events",
        F.round(F.col("predicted_rate") * 100, 3)
         .alias("predicted_rate_pct"),
        F.round(F.col("observed_rate") * 100, 3)
         .alias("observed_rate_pct"),
        F.round("gap_percentage_points", 3)
         .alias("gap_percentage_points")
    ).orderBy("population", "risk_band")
)

population,risk_band,rows,events,predicted_rate_pct,observed_rate_pct,gap_percentage_points
All eligible loans,02: 0.2% to below 0.5%,2480008,5250,0.337,0.212,0.125
All eligible loans,03: 0.5% to below 1%,643249,5691,0.664,0.885,-0.221
All eligible loans,04: 1% to below 2%,130045,2900,1.267,2.23,-0.963
All eligible loans,05: 2% to below 5%,14166,1107,3.186,7.814,-4.628
All eligible loans,06: 5% to below 10%,9580,1362,7.274,14.217,-6.943
All eligible loans,07: 10% or more,29917,8874,34.621,29.662,4.959
Current status 00,02: 0.2% to below 0.5%,2480008,5250,0.337,0.212,0.125
Current status 00,03: 0.5% to below 1%,643249,5691,0.664,0.885,-0.221
Current status 00,04: 1% to below 2%,130039,2900,1.267,2.23,-0.963
Current status 00,05: 2% to below 5%,13832,1061,3.161,7.671,-4.51
